In [1]:
import pandas as pd

df = pd.read_csv("imdb_movies_2024.csv")

print(df.head())
print(df.shape)

          Movie Name                                          Storyline  \
0      The Substance  A fading celebrity takes a black-market drug: ...   
1  The Life of Chuck  A life-affirming, genre-bending story about th...   
2          Bone Lake  A couple's vacation at a secluded estate is up...   
3              Anora  A young stripper from Brooklyn meets and impul...   
4               Eden  Based on a factual account of a group of outsi...   

                                   Cleaned_Storyline  
0  fading celebrity takes blackmarket drug cellre...  
1  lifeaffirming genrebending story three chapter...  
2  couples vacation secluded estate upended theyr...  
3  young stripper brooklyn meets impulsively marr...  
4  based factual account group outsiders settle r...  
(5099, 3)


# Dataset Inspection

In [2]:
print("Columns:")
print(df.columns)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

print("\nDataset Info:")
df.info()

Columns:
Index(['Movie Name', 'Storyline', 'Cleaned_Storyline'], dtype='object')

Missing Values:
Movie Name           0
Storyline            0
Cleaned_Storyline    0
dtype: int64

Duplicate Rows:
0

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5099 entries, 0 to 5098
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Movie Name         5099 non-null   object
 1   Storyline          5099 non-null   object
 2   Cleaned_Storyline  5099 non-null   object
dtypes: object(3)
memory usage: 119.6+ KB


# Checking few rows

In [3]:
df[['Movie Name', 'Storyline']].sample(5)

,Movie Name,Storyline
3429,Claws,Friends enjoying a camping trip in the woods e...
2344,Last Stop: Rocafort St.,A mystery that has shaken the Rocafort Metro s...
620,Hey Joe,An American veteran who got involved with a yo...
782,Freedom,"Inspired by real events, the story of Bruno Su..."
3601,Watching you - Die Welt von Palantir und Alex ...,"For the first time, a documentary examines US ..."


# Building the TF-IDF Recommendation Engine

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create TF-IDF matrix using cleaned storyline
tfidf = TfidfVectorizer(stop_words='english')

tfidf_matrix = tfidf.fit_transform(df['Cleaned_Storyline'])

print("TF-IDF Matrix Shape:")
print(tfidf_matrix.shape)

TF-IDF Matrix Shape:
(5099, 18034)


In [5]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(tfidf_matrix)

print("Cosine Similarity Matrix Shape:")
print(cosine_sim.shape)

Cosine Similarity Matrix Shape:
(5099, 5099)


# Creating the Recommendation Function

In [9]:
def recommend_movies(movie_name, cosine_sim=cosine_sim):

    idx = df[df['Movie Name'] == movie_name].index[0]

    sim_scores = list(enumerate(cosine_sim[idx]))

    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:6]

    movie_indices = [i[0] for i in sim_scores]

    recommendations = df[['Movie Name', 'Storyline']].iloc[movie_indices]

    return recommendations.reset_index(drop=True)

In [10]:
recommend_movies("The Substance")

,Movie Name,Storyline
0,The Opera! Arie per un'eclissi,Modern version of Orpheus and Eurydice.
1,Look to the Light,...Life of a wannabe celebrity influencer take...
2,A Good Man 2,"Trying to start a better life, a man after lea..."
3,The Trainer,"Follows Jack, a fitness expert living with his..."
4,The Kingdom,The story is set in the Kingdom of Kalayaan in...


# Creating Storyline-Based Recommendation Function

In [11]:
def recommend_from_storyline(user_storyline, top_n=5):
    
    # Convert user storyline into TF-IDF vector
    user_vector = tfidf.transform([user_storyline])

    # Compare with all movie storylines
    similarity_scores = cosine_similarity(user_vector, tfidf_matrix)

    # Get top matches
    top_indices = similarity_scores.argsort()[0][-top_n:][::-1]

    recommendations = df.iloc[top_indices][['Movie Name', 'Storyline']]

    return recommendations.reset_index(drop=True)

In [12]:
test_storyline = """
A young wizard begins his journey at a magical school,
where he makes friends and enemies while facing dark forces.
"""

recommend_from_storyline(test_storyline)

,Movie Name,Storyline
0,Team of Two,When facing the most dangerous criminal is eas...
1,Learning Curve,An ambitious and hardworking senior girl finds...
2,IF,A young girl who goes through a difficult expe...
3,Deaner '89,A hilarious headbanger finally makes it after ...
4,Sixty Minutes,"Desperate not to lose custody, a mixed martial..."
